In [12]:
import os

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


In [14]:
from langchain_community.vectorstores import Chroma
import numpy as np
from typing import List

In [15]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [16]:
sample_document = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

In [17]:
sample_document

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [18]:
#saving sample documents to file
import tempfile
temp_dir = tempfile.mkdtemp

In [19]:
# #iterating through the sample document list
# for i , doc in enumerate(sample_document):
#     with open(f"{temp_dir}/doc{i}.txt", "w") as file:
#         file.write(doc)

# print(f"Sample document created in {temp_dir}")

#Document Loading

In [20]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
#Load the documents from the directory

loader = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls= TextLoader,
    loader_kwargs= {'encoding' : 'utf-8'}
    )

documents = loader.load()

print(f"Loaded Documents = {len(documents)}")
print(f"\nDocument Preview")
print(f"{documents[0].page_content[:200]}....")
print(f"{documents[0].metadata}....")

Loaded Documents = 3

Document Preview

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther....
{'source': 'data\\doc_0.txt'}....


In [21]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

#Text Splitting

In [22]:
#initialize text splitters 
text_splitters = RecursiveCharacterTextSplitter(
    separators= [" "],
    chunk_size = 500, #maximum size of each chunks
    chunk_overlap = 100, #maximum overlap size of each chunks
    length_function = len,
)


In [23]:
chunks = text_splitters.split_documents(documents)
print(f"Created chunks from the documents")
print(f"Total document created {len(documents)}")
print(f"Content from first chunk {chunks[0].metadata} ---- {chunks[0].page_content[:200]}")
print(f"Total chunks created {len(chunks)}")


Created chunks from the documents
Total document created 3
Content from first chunk {'source': 'data\\doc_0.txt'} ---- Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are
Total chunks created 5


#Embeddings

In [24]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [25]:
embeddings = HuggingFaceEmbeddings()
embeddings

C:\Users\Sakshi Sinha\AppData\Local\Temp\ipykernel_41796\750672620.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\Sakshi Sinha\AppData\Local\Temp\ipykernel_41796\750672620.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1426.67it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
----------

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
vector_embeddings = embeddings.embed_query(chunks[0].page_content)
print(vector_embeddings)
print(len(vector_embeddings))

[-0.006969718728214502, -0.03977889567613602, -0.044557467103004456, -0.009834802709519863, -0.01890609785914421, 0.024438906461000443, 0.004868937656283379, 0.00854128785431385, 0.008550233207643032, -0.01099013164639473, 0.03818944841623306, 0.0793268233537674, -0.011986198835074902, 0.04592537507414818, 0.0676715075969696, -0.08811493217945099, 0.06751894950866699, -0.0682927817106247, 0.0034862742759287357, 0.010887776501476765, -0.033668145537376404, -0.018264373764395714, -0.032225195318460464, 0.009373289532959461, -0.044690437614917755, 0.0026003681123256683, -0.03829653933644295, -0.040918219834566116, -0.055867381393909454, 0.008160283789038658, 0.03210265934467316, -0.03525190427899361, 0.050896480679512024, 0.04949747771024704, 1.9895123841706663e-06, -0.04016260430216789, -0.01269868016242981, 0.0028922774363309145, 0.019503483548760414, -0.09075034409761429, 0.013272971846163273, -0.004734960850328207, 0.028848329558968544, 0.027105318382382393, -0.020307278260588646, -0.

In [27]:
#Creating a db vector store
chroma_db_dir = "./chroma_db"

#initializing the chromadb with openai embeddings
vector_stores = Chroma.from_documents(
    documents= chunks,
    embedding= embeddings,
    persist_directory= chroma_db_dir,
    collection_name="rag_Collection"
)

print(f"Vector stores has been created.. {vector_stores._collection.count()} vectors which has been persisted into {chroma_db_dir}")

Vector stores has been created.. 10 vectors which has been persisted into ./chroma_db


#Text Similarity serach

In [28]:
similar_texts = vector_stores.similarity_search(query="Deep learnings", k=3)
print(f"similar chunks/vectors related to query...")
similar_texts

similar chunks/vectors related to query...


[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

In [29]:
#detailed analysis on what contains in similar_texts
for i, doc in enumerate(similar_texts):
    print(f"{doc.metadata}")
    print(f"{doc.page_content}")

{'source': 'data\\doc_1.txt'}
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
{'source': 'data\\doc_1.txt'}
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recu

In [30]:
#Similarity search with scores
similar_texts_with_scores = vector_stores.similarity_search_with_score(query="What is deep learning", k=3)
similar_texts_with_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.5073822736740112),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recogni

In [31]:
from langchain_groq import ChatGroq
chat_model = ChatGroq(model_name="llama-3.1-8b-instant",
                       temperature=0.2,
                         max_tokens=400)
query = "What is deep learning?"

In [32]:
chat_ans = chat_model.invoke(query)
print(f"Answer from the model without context {chat_ans}")

Answer from the model without context content='Deep learning is a subset of machine learning that involves the use of artificial neural networks with multiple layers to analyze and interpret data. These neural networks are inspired by the structure and function of the human brain, with multiple layers of interconnected nodes or "neurons" that process and transmit information.\n\nDeep learning models are trained on large datasets, where the model learns to recognize patterns and relationships in the data through a process called backpropagation. This involves adjusting the model\'s parameters to minimize the difference between the predicted output and the actual output.\n\nDeep learning has several key characteristics that distinguish it from other machine learning techniques:\n\n1. **Multiple layers**: Deep learning models have multiple layers of neurons, which allow them to learn complex representations of the data.\n2. **Non-linearity**: Deep learning models use non-linear activation

Create Modern RAG Chain

In [33]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain




In [34]:
#Convert vector store into retriever
retriever =vector_stores.as_retriever(
    search_kwargs = {"k":3},

)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000020EE333A7B0>, search_kwargs={'k': 3})

In [36]:
#Creating a custom prompt
custom_prompt = """ 
You are an assistant for a question answering tasks, use the following peices of retrieved context to answer the question.
If you don't know the answer, say you don't know.
Use three sentence maximum to answer the question.

context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", custom_prompt),
        ( "human", "{input}")
    ]
)


In [37]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" \nYou are an assistant for a question answering tasks, use the following peices of retrieved context to answer the question.\nIf you don't know the answer, say you don't know.\nUse three sentence maximum to answer the question.\n\ncontext:\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [40]:
document_chain= create_stuff_documents_chain(
    llm= chat_model,
    prompt= prompt
)

In [43]:
from langchain_classic.chains import create_retrieval_chain
#Create the final RAG chain
rag_chain = create_retrieval_chain(
    retriever = retriever,
    combine_docs_chain = document_chain
    
)

Above chain takes the retrieved document
place into the prompt context placeholder
sends the complete prompt to the llm
retruns the llm response

In [44]:
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000020EE333A7B0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" \nYou are an assistant for a question answering tasks, use the following peices of retrieved context to answer the question.

In [46]:
response = rag_chain.invoke({"input": "What is deep learning?"})

In [47]:
print(response)

{'input': 'What is deep learning?', 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processin

In [48]:
response['answer']

'Deep learning is a subset of machine learning based on artificial neural networks, inspired by the human brain and consisting of layers of interconnected nodes. It has revolutionized fields like computer vision, natural language processing, and speech recognition. Deep learning is particularly effective in complex tasks that require pattern recognition and decision-making.'

In [49]:
response['context']

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neura

Create Rag chain alternative 

In [50]:
from langchain_core.structured_query import StructuredQuery
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

custom_prompt = ChatPromptTemplate.from_template(
    """ You are an AI assistant  fro a question answering tasks, use the following
    peices of retrived context to answer the question.
    If there is no relevant im=nformation in the context, say you don't know.
    Use only three sentences to answer the questions
    
    context: {context}
    Question : {question}
    answer:
    """
)
custom_prompt



ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=" You are an AI assistant  fro a question answering tasks, use the following\n    peices of retrived context to answer the question.\n    If there is no relevant im=nformation in the context, say you don't know.\n    Use only three sentences to answer the questions\n\n    context: {context}\n    Question : {question}\n    answer:\n    "), additional_kwargs={})])

In [51]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000020EE333A7B0>, search_kwargs={'k': 3})

In [ ]:
#Build the chain using LECL
